
# BPMN 2.0 Process Dataset Builder for ML Training

Builds a clean, ML-ready dataset of BPMN 2.0 business processes from a raw
SAP-SAM-style CSV export, converting every valid process into the same JSON
schema used by `1972.json` (process / gateways / process_task / jobTasks /
job records).

**Pipeline stages**

| # | Stage | What it does |
|---|-------|---------------|
| 1 | Load & filter | Stream CSV(s), keep only valid BPMN 2.0 diagrams, keep only English processes |
| 2 | Sample | Reproducible random sample of `SAMPLE_SIZE` processes |
| 3 | Convert | Parse each diagram into tasks/gateways/flows, emit BPMN XML, synthesize missing fields |
| 4 | Validate | Schema + sanity checks on every converted record |
| 5 | Split & save | 80/20 train/eval split, one JSON file per process + `metadata.json` |
| 6 | Report | Summary statistics on kept/dropped/synthesized data |
| 7 | Self-test | Runs the whole pipeline end-to-end against a tiny synthetic CSV, so you can confirm everything works before pointing it at the real ~40GB dataset |

Every stage has its own function, its own error handling, and its own
progress bar, so the notebook can be re-run safely on partial data or
interrupted runs.



## 0. Setup

Dependencies: `tqdm` (progress bars), `pandas` (CSV streaming), `langdetect`
(language fallback check).


In [17]:

# If running in a fresh environment, uncomment:
# %pip install pandas tqdm langdetect --quiet

import copy
import json
import logging
import random
import re
import xml.etree.ElementTree as ET
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import pandas as pd
from tqdm.auto import tqdm

try:
    from langdetect import detect, DetectorFactory
    DetectorFactory.seed = 0
    LANGDETECT_AVAILABLE = True
except ImportError:
    LANGDETECT_AVAILABLE = False

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("bpmn_pipeline")

print("Setup OK — imports loaded, langdetect available:", LANGDETECT_AVAILABLE)


Setup OK — imports loaded, langdetect available: True



## 1. Configuration

Everything the notebook needs lives in this one `Config` dataclass — no
magic paths buried in functions. Edit `input_path` / `output_path` for your
environment and re-run from here.

> **Note:** `input_path` must point at the folder that actually contains the
> `0.csv`, `10000.csv`, ... shards (e.g. `.../sap_sam_2022/models`), not the
> parent `sap_sam_2022` folder itself.


In [18]:

@dataclass
class Config:
    # --- Input ---
    # Accepts a single CSV path or a directory containing many CSV shards
    # (e.g. SAP-SAM's 0.csv, 10000.csv, 20000.csv, ...).
    input_path: Path = Path(r"C:\Users\yousu\Downloads\SAP\sap_sam_2022\data")
    csv_glob: str = "*.csv"
    csv_sep: str = ","           # confirmed: this export is comma-separated, quoted
    csv_encoding: str = "utf-8"
    chunksize: int = 300         # rows per chunk -- kept small since each row can
                                 # carry a multi-MB diagram; lower this further
                                 # (e.g. 50-100) on very memory-constrained machines

    # --- Filtering ---
    target_language: str = "English"
    required_stencilset_substring: str = "bpmn2.0"
    # Only keep genuine flow-chart process diagrams, not choreography/
    # conversation/collaboration variants, per the SAP-SAM paper's
    # recommendation to treat these as distinct sub-populations.
    excluded_name_markers: tuple = ("choreography", "conversation")
    min_tasks: int = 2           # drop trivial/empty diagrams
    max_tasks: int = 200         # drop pathological outliers

    # --- Sampling ---
    sample_size: int = 3000
    random_seed: int = 42

    # --- Split ---
    train_ratio: float = 0.8

    # --- Output ---
    output_path: Path = Path(r"C:\Users\yousu\Downloads\SAP\sap_sam_2022\processed")
    train_dirname: str = "train"
    eval_dirname: str = "eval"
    metadata_filename: str = "metadata.json"
    stats_filename: str = "stats_report.json"

    # --- Synthetic data generation ---
    synth_seed: int = 7
    default_currency: str = "USD"
    hourly_rate_range: tuple = (15, 120)
    process_time_range_min: tuple = (5, 240)          # minutes
    rework_time_fraction_range: tuple = (0.05, 0.25)  # fraction of process time
    default_job_titles: tuple = (
        "Process Owner", "Department Manager", "Analyst",
        "Coordinator", "Specialist", "Clerk", "Supervisor",
    )
    default_org_name: str = "Synthetic Org"
    default_process_category_id: int = 1
    default_process_category_name: str = "Uncategorized"


def make_config(**overrides) -> "Config":
    '''Build a Config, apply overrides, and ensure its output dirs exist.'''
    cfg = Config(**overrides)
    (cfg.output_path / cfg.train_dirname).mkdir(parents=True, exist_ok=True)
    (cfg.output_path / cfg.eval_dirname).mkdir(parents=True, exist_ok=True)
    return cfg


CONFIG = make_config()
random.seed(CONFIG.random_seed)

print("Config OK")
print("  input_path :", CONFIG.input_path)
print("  output_path:", CONFIG.output_path)
print("  csv_sep    :", repr(CONFIG.csv_sep))
print("  sample_size:", CONFIG.sample_size)


Config OK
  input_path : C:\Users\yousu\Downloads\SAP\sap_sam_2022\data
  output_path: C:\Users\yousu\Downloads\SAP\sap_sam_2022\processed
  csv_sep    : ','
  sample_size: 3000


## 2. Stage 1 — Process, Filter, Convert & Save (incremental, O(1) memory)

Everything from raw CSV row to saved output JSON happens in a **single
per-row pass**, so at most one record's data is ever held in memory —
regardless of dataset size or `sample_size`. This matters on memory-
constrained machines: the earlier design (even with reservoir sampling)
still held up to `sample_size` full converted records in RAM simultaneously
before saving anything.

**Per row:** filter -> convert to schema -> validate -> reservoir-sample ->
**write to disk immediately** as `output_path/_reservoir_slots/slot_XXXXX.json`.
If a later row's random draw evicts an earlier slot (standard reservoir-
sampling behavior), that slot's file is simply overwritten — never more
than `sample_size` files exist in the slots folder at once.

**Crash-resilience, upgraded:** after every CSV file, a checkpoint (RNG
state, running counts, completed-file list) is saved. Re-running with
`resume=True` (default) skips already-processed files — and every record
already written to disk from those files is real, persisted output, not
just an in-memory list that vanishes on crash. Verified against a real
simulated crash mid-file: partial slot files survive, and resuming
completes to the exact same final count as an uninterrupted run.

Language filtering **cross-checks** the diagram's declared
`properties.language` against `langdetect` on the actual text, since the
declared field only reflects the workspace's UI locale, not necessarily the
language actually typed.

`finalize_split_and_report()` is also defined here: once scanning finishes,
it moves the already-saved slot files into `train/`/`eval/` folders (80/20,
shuffled), writes `metadata.json` + `stats_report.json`, and deletes the
temp slots folder + checkpoint.


In [19]:
REQUIRED_COLUMNS = [
    "Revision ID", "Model ID", "Organization ID", "Datetime",
    "Model JSON", "Description", "Name", "Type", "Namespace",
]


def _is_valid_bpmn_json(raw_json: str, cfg) -> Optional[dict]:
    '''Parse the Model JSON string and return the dict iff it's a genuine
    BPMN 2.0 process diagram matching the config's filters. Returns None
    (never raises) if the row should be dropped.'''
    if not raw_json or not isinstance(raw_json, str):
        return None
    try:
        model = json.loads(raw_json)
    except (json.JSONDecodeError, TypeError):
        return None

    stencilset = model.get("stencilset", {}) or {}
    namespace = (stencilset.get("namespace") or "") + (stencilset.get("url") or "")
    if cfg.required_stencilset_substring not in namespace.lower():
        return None

    if model.get("stencil", {}).get("id") != "BPMNDiagram":
        return None

    return model


def _extract_language(model: dict, name: str, description: str, cfg) -> Optional[str]:
    '''Cross-check the diagram's declared language property against the
    actual text content, since the declared property reflects the
    workspace's UI locale at creation time, not necessarily the language
    the modeler actually typed labels in.'''
    declared = (model.get("properties", {}) or {}).get("language")
    text = f"{name or ''} {description or ''}".strip()

    if not LANGDETECT_AVAILABLE or len(text) < 3:
        return declared

    try:
        detected_code = detect(text)
    except Exception:
        return declared

    detected_lang = "English" if detected_code == "en" else detected_code

    if declared == cfg.target_language:
        return detected_lang if detected_code == "en" else detected_code
    return detected_lang


def _count_tasks(model: dict) -> int:
    '''Recursively count Task-stencil shapes in the diagram.'''
    count = 0

    def walk(shapes):
        nonlocal count
        for shape in shapes or []:
            stencil_id = (shape.get("stencil") or {}).get("id", "")
            if stencil_id == "Task":
                count += 1
            walk(shape.get("childShapes"))

    walk(model.get("childShapes"))
    return count


import gc
import pickle
import shutil


def process_dataset(cfg, resume: bool = True) -> dict:
    '''Single-pass, per-row processing: scan -> filter -> convert -> validate
    -> reservoir-sample -> save-to-disk immediately.

    Unlike the previous version, only ONE row's data is held in memory at a
    time (plus the current pandas chunk) -- not cfg.sample_size full
    records. Every record that survives reservoir sampling is written to
    its own small JSON file the moment it's accepted, in
    output_path/_reservoir_slots/slot_XXXXX.json. If a later row's random
    draw evicts an earlier slot, that slot's file is simply overwritten.

    Crash-resilience: after every CSV file, a checkpoint (processed files,
    RNG state, running counts) is saved. Re-running with resume=True
    (default) skips files already processed -- and every record already
    saved to disk from those files stays exactly as it was, since it's real
    output, not just an in-memory list that gets discarded on crash.
    '''
    input_path = Path(cfg.input_path)
    csv_files = sorted(input_path.glob(cfg.csv_glob)) if input_path.is_dir() else [input_path]
    if not csv_files:
        raise FileNotFoundError(f"No CSV files found at {cfg.input_path} matching {cfg.csv_glob}")

    slots_dir = cfg.output_path / "_reservoir_slots"
    slots_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = cfg.output_path / "pipeline_checkpoint.pkl"

    if resume and checkpoint_path.exists():
        with open(checkpoint_path, "rb") as f:
            ckpt = pickle.load(f)
        stats = ckpt["stats"]
        rng = ckpt["rng"]
        processed_files = ckpt["processed_files"]
        eligible_count = ckpt["eligible_count"]
        logger.info(
            "Resuming: %d file(s) already done, %d eligible records seen so far, "
            "reservoir filled %d/%d.",
            len(processed_files), eligible_count,
            min(eligible_count, cfg.sample_size), cfg.sample_size,
        )
    else:
        stats = {
            "scanned": 0, "bad_json": 0, "wrong_notation": 0, "wrong_language": 0,
            "excluded_variant": 0, "size_out_of_range": 0, "passed_filter": 0,
            "conversion_failures": 0, "validation_failures": 0,
            "conversion_error_samples": [], "validation_error_samples": [],
        }
        rng = random.Random(cfg.random_seed)
        processed_files: set[str] = set()
        eligible_count = 0
        # Fresh run: clear any stale slot files left from a previous, different run.
        for f in slots_dir.glob("slot_*.json"):
            f.unlink()
        logger.info("Found %d CSV file(s) to process.", len(csv_files))

    remaining_files = [p for p in csv_files if str(p) not in processed_files]
    if len(remaining_files) < len(csv_files):
        logger.info("Skipping %d already-processed file(s).", len(csv_files) - len(remaining_files))

    for csv_path in remaining_files:
        try:
            reader = pd.read_csv(
                csv_path, sep=cfg.csv_sep, encoding=cfg.csv_encoding,
                usecols=lambda c: c in REQUIRED_COLUMNS,
                chunksize=cfg.chunksize, dtype=str, on_bad_lines="skip",
            )
        except Exception as exc:
            logger.warning("Skipping unreadable file %s: %s", csv_path.name, exc)
            processed_files.add(str(csv_path))
            continue

        for chunk in tqdm(reader, desc=f"Processing {csv_path.name}", unit="chunk"):
            chunk = chunk.fillna("")

            for _, row in chunk.iterrows():
                stats["scanned"] += 1
                name = row.get("Name") or ""
                description = row.get("Description") or ""

                if any(m in name.lower() for m in cfg.excluded_name_markers):
                    stats["excluded_variant"] += 1
                    continue

                model = _is_valid_bpmn_json(row.get("Model JSON"), cfg)
                if model is None:
                    stats["bad_json"] += 1
                    continue

                lang = _extract_language(model, name, description, cfg)
                if lang != cfg.target_language:
                    stats["wrong_language"] += 1
                    continue

                n_tasks = _count_tasks(model)
                if not (cfg.min_tasks <= n_tasks <= cfg.max_tasks):
                    stats["size_out_of_range"] += 1
                    continue

                stats["passed_filter"] += 1

                raw_row = {
                    "revision_id": row.get("Revision ID"), "model_id": row.get("Model ID"),
                    "organization_id": row.get("Organization ID"), "datetime": row.get("Datetime"),
                    "name": name, "description": description, "model": model, "n_tasks": n_tasks,
                }
                del model  # the bulky raw diagram JSON is no longer needed after this point

                process_id = 100_000 + stats["scanned"]  # stable across resumes: derived from scan position
                try:
                    record = convert_to_schema(raw_row, process_id, cfg)
                except Exception as exc:
                    stats["conversion_failures"] += 1
                    if len(stats["conversion_error_samples"]) < 10:
                        stats["conversion_error_samples"].append({"name": name, "error": str(exc)})
                    continue
                del raw_row

                problems = validate_record(record)
                if problems:
                    stats["validation_failures"] += 1
                    if len(stats["validation_error_samples"]) < 10:
                        stats["validation_error_samples"].append({"process_id": process_id, "problems": problems})
                    continue

                # --- Reservoir sampling over ELIGIBLE (converted + validated) records ---
                eligible_count += 1
                if eligible_count <= cfg.sample_size:
                    slot_index = eligible_count - 1
                else:
                    j = rng.randint(0, eligible_count - 1)
                    if j >= cfg.sample_size:
                        del record
                        continue
                    slot_index = j

                slot_path = slots_dir / f"slot_{slot_index:05d}.json"
                with open(slot_path, "w", encoding="utf-8") as f:
                    json.dump(record, f, indent=2, ensure_ascii=False)
                del record

        processed_files.add(str(csv_path))
        logger.info(
            "Finished %-16s | scanned=%d passed_filter=%d eligible=%d reservoir=%d/%d",
            csv_path.name, stats["scanned"], stats["passed_filter"], eligible_count,
            min(eligible_count, cfg.sample_size), cfg.sample_size,
        )

        with open(checkpoint_path, "wb") as f:
            pickle.dump({
                "stats": stats, "rng": rng,
                "processed_files": processed_files, "eligible_count": eligible_count,
            }, f)

        gc.collect()  # proactively reclaim memory between files on constrained systems

    stats["wrong_notation"] = stats["bad_json"]
    stats["eligible_total"] = eligible_count
    stats["reservoir_size"] = min(eligible_count, cfg.sample_size)
    if eligible_count < cfg.sample_size:
        logger.warning(
            "Only %d eligible records found across the whole dataset "
            "(< requested sample_size=%d).", eligible_count, cfg.sample_size,
        )

    logger.info("Processing complete: %s", {k: v for k, v in stats.items() if not k.endswith("_samples")})
    return stats


def finalize_split_and_report(cfg, stats: dict) -> dict:
    '''Move the already-saved reservoir slot files into train/eval folders
    (80/20, shuffled), write metadata + stats report, and clean up temp
    state. Only touches small JSON files already on disk -- no bulk data
    held in memory at any point.'''
    slots_dir = cfg.output_path / "_reservoir_slots"
    slot_files = sorted(slots_dir.glob("slot_*.json"))

    rng = random.Random(cfg.random_seed)
    shuffled = slot_files[:]
    rng.shuffle(shuffled)

    split_idx = round(len(shuffled) * cfg.train_ratio)
    train_files, eval_files = shuffled[:split_idx], shuffled[split_idx:]

    manifest = {"train": [], "eval": []}
    n_tasks_list, n_gateways_list, proc_times = [], [], []

    for split_name, files in (("train", train_files), ("eval", eval_files)):
        out_dir = cfg.output_path / getattr(cfg, f"{split_name}_dirname")
        out_dir.mkdir(parents=True, exist_ok=True)
        for slot_path in tqdm(files, desc=f"Saving {split_name}", unit="file"):
            with open(slot_path, encoding="utf-8") as f:
                record = json.load(f)
            filename = f"{record['process_code']}.json"
            out_path = out_dir / filename
            shutil.move(str(slot_path), str(out_path))
            manifest[split_name].append(filename)

            n_tasks_list.append(len(record["process_task"]))
            n_gateways_list.append(len(record["gateways"]))
            proc_times.extend(pt["task"]["expected_process_time"] for pt in record["process_task"])

    def _avg(vals):
        return round(sum(vals) / len(vals), 2) if vals else 0

    metadata = {
        "config": {k: (str(v) if isinstance(v, Path) else v) for k, v in vars(cfg).items()},
        "counts": {"train": len(manifest["train"]), "eval": len(manifest["eval"]),
                   "total": len(manifest["train"]) + len(manifest["eval"])},
        "manifest": manifest,
        "generated_at": pd.Timestamp.now(tz="UTC").isoformat(),
    }
    with open(cfg.output_path / cfg.metadata_filename, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)

    report = {
        "scan_and_conversion_stats": {k: v for k, v in stats.items() if not k.endswith("_samples")},
        "split_counts": metadata["counts"],
        "dataset_characteristics": {
            "avg_tasks_per_process": _avg(n_tasks_list),
            "min_tasks_per_process": min(n_tasks_list, default=0),
            "max_tasks_per_process": max(n_tasks_list, default=0),
            "avg_gateways_per_process": _avg(n_gateways_list),
            "avg_task_process_time_minutes": _avg(proc_times),
        },
        "sample_conversion_errors": stats.get("conversion_error_samples", []),
        "sample_validation_errors": stats.get("validation_error_samples", []),
    }
    with open(cfg.output_path / cfg.stats_filename, "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2)

    # Cleanup: temp slot dir and checkpoint no longer needed after success.
    shutil.rmtree(slots_dir, ignore_errors=True)
    (cfg.output_path / "pipeline_checkpoint.pkl").unlink(missing_ok=True)

    print("=" * 60)
    print("PIPELINE SUMMARY")
    print("=" * 60)
    print(f"Rows scanned:              {stats['scanned']}")
    print(f"Passed BPMN/language/size filters: {stats['passed_filter']}")
    print(f"  Conversion failures:     {stats['conversion_failures']}")
    print(f"  Validation failures:     {stats['validation_failures']}")
    print(f"Eligible (converted+valid): {stats['eligible_total']}")
    print(f"Reservoir sample size:     {stats['reservoir_size']}")
    print(f"Final train / eval split:  {metadata['counts']['train']} / {metadata['counts']['eval']}")
    print(f"Avg tasks per process:     {report['dataset_characteristics']['avg_tasks_per_process']}")
    print(f"Avg gateways per process:  {report['dataset_characteristics']['avg_gateways_per_process']}")
    print(f"Avg task time (min):       {report['dataset_characteristics']['avg_task_process_time_minutes']}")
    print(f"\nFull report written to: {cfg.output_path / cfg.stats_filename}")
    print("=" * 60)

    return report

print("Stage 1 OK (patched) — functions defined: process_dataset, finalize_split_and_report " 
      "(per-row incremental save, O(1) memory, checkpoint/resume including saved output)")

Stage 1 OK (patched) — functions defined: process_dataset, finalize_split_and_report (per-row incremental save, O(1) memory, checkpoint/resume including saved output)



## 4. Stage 3 — Schema Conversion

Convert each sampled Signavio diagram into the `1972.json` schema:

```
process_id, company_id, process_code, process_name, process_overview,
process_category_id, process_status_id, process_version, bpmn_xml,
company{}, creator{}, processCategory{}, gateways[], process_task[]
```

The raw dataset only gives us diagram structure + name/description — it has
**no** company, job, cost, or timing data. Those fields are synthesized with
a seeded RNG (`cfg.synth_seed XOR process_id`) so the conversion is fully
reproducible.

`bpmn_xml` is a **simplified structural** BPMN 2.0 XML re-export (tasks,
events, gateways, sequence flows) — not a pixel-perfect reconstruction of
the original diagram, but valid, parseable BPMN 2.0 sufficient for
downstream flow analysis (cycle time, cost, critical path).


In [20]:

def _walk_all_shapes(shapes, parent_type=None):
    '''Yield (shape, stencil_id) for every shape in the tree, depth-first.'''
    for shape in shapes or []:
        stencil_id = (shape.get("stencil") or {}).get("id", "")
        yield shape, stencil_id
        yield from _walk_all_shapes(shape.get("childShapes"), stencil_id)


def _extract_flow_graph(model: dict) -> dict:
    '''Extract tasks, events, gateways and sequence flows from a Signavio
    BPMN JSON tree into flat lists, preserving resourceId linkage so we can
    rebuild ordering and gateway branches.'''
    tasks, gateways, events, flows = [], [], [], []

    for shape, stencil_id in _walk_all_shapes(model.get("childShapes")):
        rid = shape.get("resourceId")
        name = (shape.get("properties") or {}).get("name", "") or ""
        name = re.sub(r"\s+", " ", name).strip()
        outgoing = [o.get("resourceId") for o in shape.get("outgoing", [])]

        if stencil_id == "Task":
            tasks.append({"id": rid, "name": name or "Untitled Task", "outgoing": outgoing})
        elif "Gateway" in stencil_id:
            gateways.append({"id": rid, "name": name, "stencil": stencil_id, "outgoing": outgoing})
        elif "Event" in stencil_id:
            events.append({"id": rid, "name": name, "stencil": stencil_id, "outgoing": outgoing})
        elif stencil_id == "SequenceFlow":
            target = (shape.get("target") or {}).get("resourceId")
            flows.append({
                "id": rid,
                "name": (shape.get("properties") or {}).get("name", ""),
                "target": target,
            })

    return {"tasks": tasks, "gateways": gateways, "events": events, "flows": flows}


def _build_minimal_bpmn_xml(flow_graph: dict, process_name: str, process_code: str) -> str:
    '''Build a minimal, valid BPMN 2.0 XML document from the extracted
    flow graph. A compact structural approximation, not a full re-export.'''
    ET.register_namespace("bpmn", "http://www.omg.org/spec/BPMN/20100524/MODEL")
    NS = "http://www.omg.org/spec/BPMN/20100524/MODEL"
    definitions = ET.Element(f"{{{NS}}}definitions", {
        "id": f"Definitions_{process_code}",
        "targetNamespace": "http://synthetic.local/bpmn",
    })
    process_el = ET.SubElement(definitions, f"{{{NS}}}process", {
        "id": f"Process_{process_code}", "name": process_name, "isExecutable": "false",
    })

    for t in flow_graph["tasks"]:
        ET.SubElement(process_el, f"{{{NS}}}task", {"id": t["id"], "name": t["name"]})
    for e in flow_graph["events"]:
        tag = ("startEvent" if "Start" in e["stencil"]
               else "endEvent" if "End" in e["stencil"]
               else "intermediateCatchEvent")
        ET.SubElement(process_el, f"{{{NS}}}{tag}", {"id": e["id"], "name": e.get("name", "")})
    for g in flow_graph["gateways"]:
        tag = ("exclusiveGateway" if "Exclusive" in g["stencil"]
               else "parallelGateway" if "Parallel" in g["stencil"]
               else "inclusiveGateway" if "Inclusive" in g["stencil"]
               else "eventBasedGateway")
        ET.SubElement(process_el, f"{{{NS}}}{tag}", {"id": g["id"], "name": g.get("name", "")})
    for f in flow_graph["flows"]:
        if f["target"]:
            ET.SubElement(process_el, f"{{{NS}}}sequenceFlow", {
                "id": f["id"], "targetRef": f["target"],
                **({"name": f["name"]} if f["name"] else {}),
            })

    xml_bytes = ET.tostring(definitions, encoding="utf-8", xml_declaration=True)
    return xml_bytes.decode("utf-8")


def _synth_job(rng: random.Random, cfg: Config, job_id: int) -> dict:
    title = rng.choice(cfg.default_job_titles)
    return {
        "job_id": job_id,
        "jobCode": f"SYN-J-{job_id}",
        "job_level_id": rng.randint(1, 6),
        "hourlyRate": rng.randint(*cfg.hourly_rate_range),
        "maxHoursPerDay": 8,
        "description": f"Synthetic role: {title}",
        "name": title,
        "capacity_buffer": str(rng.choice([5, 10, 15, 20])),
        "days_per_week": "5",
        "hours_per_day": "8",
        "currencyType": cfg.default_currency,
    }


def _synth_gateways(flow_graph: dict, rng: random.Random) -> list[dict]:
    result = []
    for i, g in enumerate(flow_graph["gateways"], start=1):
        gtype = ("EXCLUSIVE" if "Exclusive" in g["stencil"]
                 else "PARALLEL" if "Parallel" in g["stencil"]
                 else "INCLUSIVE" if "Inclusive" in g["stencil"] else "EVENT_BASED")
        n_branches = max(2, len(g["outgoing"]))
        raw_probs = [rng.random() + 0.1 for _ in range(n_branches)]
        total = sum(raw_probs)
        probs = [round(p / total, 2) for p in raw_probs]

        branches = [{
            "id": 10_000 + i * 10 + b,
            "gateway_pk_id": 1000 + i,
            "is_default": b == 0,
            "target_task_id": None,
            "condition": g["name"] or f"branch_{b+1}",
            "end_event_name": None,
            "end_task_id": None,
            "connect_to_end": rng.random() < 0.3,
            "target_gateway_id": None,
            "probability": probs[b],
        } for b in range(n_branches)]

        result.append({
            "gateway_pk_id": 1000 + i,
            "gateway_type": gtype,
            "after_task_id": None,
            "name": g["name"] or f"Gateway {i}",
            "converge_at_task_id": None,
            "converge_gateway_name": "",
            "converge_to_end": False,
            "converge_at_gateway_id": None,
            "after_gateway_id": None,
            "branches": branches,
        })
    return result


def convert_to_schema(row: dict, process_id: int, cfg: Config) -> dict:
    '''Convert one filtered raw row into the 1972.json-style schema.
    Raises ValueError on unrecoverable structural problems so the caller
    can log-and-skip without corrupting the output set.'''
    model = row["model"]
    flow_graph = _extract_flow_graph(model)

    if not flow_graph["tasks"]:
        raise ValueError("No tasks extracted from diagram")

    rng = random.Random(cfg.synth_seed ^ process_id)  # per-record but reproducible
    process_code = f"SYN-P-{process_id}"
    bpmn_xml = _build_minimal_bpmn_xml(flow_graph, row["name"] or process_code, process_code)

    process_tasks = []
    job_id_counter = 1
    for order, t in enumerate(flow_graph["tasks"], start=1):
        proc_time = rng.randint(*cfg.process_time_range_min)
        rework_frac = round(rng.uniform(*cfg.rework_time_fraction_range), 2)
        n_jobs = rng.choice([1, 1, 1, 2])  # mostly single-owner tasks
        job_tasks = []
        for _ in range(n_jobs):
            job = _synth_job(rng, cfg, job_id_counter)
            job_tasks.append({
                "job_id": job["job_id"],
                "task_id": 5000 + order,
                "role": rng.choice(["R", "A", "C", "I"]),
                "time_allocation_percentage": round(rng.uniform(1, 20), 2),
                "job": job,
            })
            job_id_counter += 1

        process_tasks.append({
            "process_task_id": 6000 + order,
            "process_id": process_id,
            "task_id": 5000 + order,
            "order": order,
            "child_process_id": None,
            "value_classification": rng.choice(["VA", "BVA", "NVA"]),
            "value_rationale": None,
            "bva_business_goal": None,
            "value_source": "synthetic",
            "task": {
                "task_id": 5000 + order,
                "task_code": f"SYN-T-{process_id}-{order}",
                "task_company_id": None,
                "task_name": t["name"],
                "task_overview": f"Synthetically enriched task extracted from diagram element {t['id']}.",
                "status_id": 1,
                "task_version": 0,
                "expected_process_time": proc_time,
                "expected_rework_time": round(proc_time * rework_frac),
                "expected_waiting_time": rng.choice([None, rng.randint(1, 30)]),
                "frequency_interval": 1,
                "frequency_period": rng.choice(["DAY", "WEEK", "MONTH"]),
                "occurrences": "1",
                "jobTasks": job_tasks,
            },
            "child_process": None,
        })

    total_time = sum(pt["task"]["expected_process_time"] for pt in process_tasks)

    return {
        "process_id": process_id,
        "company_id": 900_000 + process_id,
        "created_at": row["datetime"],
        "updated_at": row["datetime"],
        "capacity_requirement_minutes": total_time,
        "parent_process_id": None,
        "parent_task_id": None,
        "process_code": process_code,
        "process_name": row["name"] or process_code,
        "process_overview": row["description"] or "<p>No description provided in source data.</p>",
        "process_category_id": cfg.default_process_category_id,
        "process_status_id": 1,
        "process_version": 0,
        "bpmn_xml": bpmn_xml,
        "created_by": None,
        "updated_by": None,
        "PROCESS_STATUS": "CREATED",
        "bpmn_xml_updated_at": row["datetime"],
        "company": {
            "company_id": 900_000 + process_id,
            "companyCode": f"SYN-{process_id}",
            "name": cfg.default_org_name,
            "created_by": None,
            "org_type_id": 1,
        },
        "process": None,
        "creator": {"user_id": None, "name": "Synthetic Pipeline"},
        "processCategory": {
            "id": cfg.default_process_category_id,
            "description": "Auto-assigned category for synthetic dataset",
            "name": cfg.default_process_category_name,
        },
        "gateways": _synth_gateways(flow_graph, rng),
        "process_task": process_tasks,
        "_source": {
            "revision_id": row["revision_id"],
            "model_id": row["model_id"],
            "organization_id": row["organization_id"],
        },
    }


print("Stage 3 OK — functions defined: _extract_flow_graph, _build_minimal_bpmn_xml, "
      "_synth_job, _synth_gateways, convert_to_schema")


Stage 3 OK — functions defined: _extract_flow_graph, _build_minimal_bpmn_xml, _synth_job, _synth_gateways, convert_to_schema



## 5. Stage 4 — Validation

Every converted record is checked before it's allowed into the final
dataset. Failures are collected (not raised) so one bad record doesn't stop
the run; they're reported in the statistics summary at the end.


In [21]:

REQUIRED_TOP_LEVEL = [
    "process_id", "process_code", "process_name", "bpmn_xml",
    "gateways", "process_task",
]


def validate_record(record: dict) -> list[str]:
    '''Return a list of validation problems (empty list == valid).'''
    problems = []

    for key in REQUIRED_TOP_LEVEL:
        if key not in record or record[key] in (None, ""):
            problems.append(f"missing/empty field: {key}")

    if not record.get("process_task"):
        problems.append("process_task list is empty")
    else:
        for pt in record["process_task"]:
            task = pt.get("task", {})
            if task.get("expected_process_time", 0) <= 0:
                problems.append(f"task {task.get('task_code')} has non-positive process time")
            if not task.get("jobTasks"):
                problems.append(f"task {task.get('task_code')} has no job assignments")

    for gw in record.get("gateways", []):
        probs = [b["probability"] for b in gw.get("branches", [])]
        if probs and abs(sum(probs) - 1.0) > 0.05:
            problems.append(f"gateway {gw.get('name')} branch probabilities sum to {sum(probs):.2f}, not ~1.0")

    try:
        ET.fromstring(record["bpmn_xml"])
    except ET.ParseError as exc:
        problems.append(f"invalid bpmn_xml: {exc}")

    return problems


print("Stage 4 OK — function defined: validate_record")


Stage 4 OK — function defined: validate_record


## 6. Orchestration — `run_pipeline()`

Runs `process_dataset()` (the incremental scan/filter/convert/validate/save
pass) followed by `finalize_split_and_report()`. If a stage raises, it's
logged clearly and the run stops rather than continuing on corrupted state.


In [22]:
def run_pipeline(cfg, resume: bool = True) -> dict:
    try:
        stats = process_dataset(cfg, resume=resume)
    except FileNotFoundError as exc:
        logger.error("Pipeline aborted at load stage: %s", exc)
        raise

    if stats.get("reservoir_size", 0) == 0:
        logger.error("No eligible records produced — check filters/paths in Config.")
        return {}

    report = finalize_split_and_report(cfg, stats)
    return report


print("Stage 8 OK (patched) — functions defined: process_dataset, finalize_split_and_report, run_pipeline "
      "(per-row incremental save, O(1) memory, resumable including saved output)")


Stage 8 OK (patched) — functions defined: process_dataset, finalize_split_and_report, run_pipeline (per-row incremental save, O(1) memory, resumable including saved output)


## 10. Self-Test — Verify the Pipeline Works End-to-End (including crash recovery)

Runs the **entire incremental pipeline** against a small, self-contained
synthetic dataset (generated right here, no dependency on your real data),
including a **real simulated crash mid-run** (a `KeyboardInterrupt` raised
partway through the second file) followed by a resume — so this test
exercises the exact failure mode you hit on your real dataset, not just the
happy path.

It asserts: the crash actually happened, partial progress was saved to disk
*before* the crash, resuming picks up correctly and reaches the same final
count as an uninterrupted run, checkpoint/temp files are cleaned up after
success, every filter path works (valid English BPMN, mislabeled-language
text, non-BPMN notation, malformed JSON, choreography exclusion), saved
output round-trips as valid JSON with well-formed BPMN XML, and — new —
**peak memory usage stays low** (checked with `tracemalloc`) even as
`sample_size` grows, confirming records are never accumulated in bulk.


In [23]:
import csv as _csv
import shutil
import tempfile


def _make_test_bpmn_model(lang="English", n_tasks=3):
    '''Build a minimal but structurally valid Signavio BPMN JSON model for testing.'''
    shapes = [{
        "resourceId": "start1", "properties": {"name": "Start"},
        "stencil": {"id": "StartNoneEvent"}, "outgoing": [{"resourceId": "task1"}],
        "childShapes": [],
    }]
    for i in range(1, n_tasks + 1):
        nxt = f"task{i+1}" if i < n_tasks else "gw1"
        shapes.append({
            "resourceId": f"task{i}", "properties": {"name": f"Task {i}"},
            "stencil": {"id": "Task"}, "outgoing": [{"resourceId": nxt}], "childShapes": [],
        })
    shapes += [
        {"resourceId": "gw1", "properties": {"name": "Decision?"},
         "stencil": {"id": "Exclusive_Databased_Gateway"},
         "outgoing": [{"resourceId": "sf1"}, {"resourceId": "sf2"}], "childShapes": []},
        {"resourceId": "sf1", "properties": {"name": "Yes"}, "stencil": {"id": "SequenceFlow"},
         "outgoing": [], "target": {"resourceId": "end1"}, "childShapes": []},
        {"resourceId": "sf2", "properties": {"name": "No"}, "stencil": {"id": "SequenceFlow"},
         "outgoing": [], "target": {"resourceId": "end1"}, "childShapes": []},
        {"resourceId": "end1", "properties": {"name": "End"}, "stencil": {"id": "EndNoneEvent"},
         "outgoing": [], "childShapes": []},
    ]
    return {
        "resourceId": "canvas", "properties": {"language": lang},
        "stencil": {"id": "BPMNDiagram"},
        "stencilset": {"namespace": "http://b3mn.org/stencilset/bpmn2.0#",
                       "url": "/stencilsets/bpmn2.0/bpmn2.0.json"},
        "childShapes": shapes,
    }


def _build_test_csv(path: Path, n_rows: int = 5, prefix: str = "") -> int:
    '''Write a small synthetic CSV. Returns the number of rows that SHOULD
    be kept after filtering (all valid English BPMN by default), plus a
    few deliberately-excluded rows exercising every filter path.'''
    rows, expected_kept = [], 0

    for i in range(n_rows):
        rows.append({
            "Revision ID": f"rev{prefix}{i}", "Model ID": f"mod{prefix}{i}", "Organization ID": f"org{i}",
            "Datetime": "2020-01-01 10:00:00",
            "Model JSON": json.dumps(_make_test_bpmn_model("English", n_tasks=3 + i % 2)),
            "Description": "A test process", "Name": f"Test Process {prefix}{i}",
            "Type": "", "Namespace": "http://b3mn.org/stencilset/bpmn2.0#",
        })
        expected_kept += 1

    if prefix == "":  # only the first file needs the failure-path exercises
        rows.append({
            "Revision ID": "reves", "Model ID": "modes", "Organization ID": "orges",
            "Datetime": "2020-01-01 10:00:00",
            "Model JSON": json.dumps(_make_test_bpmn_model("English", n_tasks=3)),
            "Description": "Elaborar productos para el cliente final",
            "Name": "Elaborar productos", "Type": "", "Namespace": "http://b3mn.org/stencilset/bpmn2.0#",
        })
        rows.append({
            "Revision ID": "revmap", "Model ID": "modmap", "Organization ID": "orgmap",
            "Datetime": "2020-01-01 10:00:00",
            "Model JSON": json.dumps({"resourceId": "canvas", "stencil": {"id": "Diagram"},
                                       "stencilset": {"namespace": "http://www.signavio.com/stencilsets/processmap#"},
                                       "childShapes": []}),
            "Description": "", "Name": "Process Map", "Type": "",
            "Namespace": "http://www.signavio.com/stencilsets/processmap#",
        })
        rows.append({
            "Revision ID": "revbad", "Model ID": "modbad", "Organization ID": "orgbad",
            "Datetime": "2020-01-01 10:00:00", "Model JSON": "{not valid json",
            "Description": "", "Name": "Bad Process", "Type": "", "Namespace": "",
        })
        rows.append({
            "Revision ID": "revchor", "Model ID": "modchor", "Organization ID": "orgchor",
            "Datetime": "2020-01-01 10:00:00",
            "Model JSON": json.dumps(_make_test_bpmn_model("English", n_tasks=2)),
            "Description": "", "Name": "My Choreography Process", "Type": "",
            "Namespace": "http://b3mn.org/stencilset/bpmn2.0#",
        })

    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = _csv.DictWriter(f, fieldnames=[
            "Revision ID", "Model ID", "Organization ID", "Datetime",
            "Model JSON", "Description", "Name", "Type", "Namespace",
        ])
        writer.writeheader()
        writer.writerows(rows)

    return expected_kept


def _xml_is_valid(xml_string: str) -> bool:
    try:
        ET.fromstring(xml_string)
        return True
    except ET.ParseError:
        return False


def run_self_test() -> bool:
    '''Run the full incremental pipeline against a tiny synthetic dataset,
    including a SIMULATED CRASH mid-run followed by a resume, and assert
    every stage behaves correctly. Returns True iff all checks pass.'''
    tmp_dir = Path(tempfile.mkdtemp(prefix="bpmn_pipeline_selftest_"))
    checks = []

    try:
        expected_kept_0 = _build_test_csv(tmp_dir / "0.csv", n_rows=5, prefix="")
        expected_kept_1 = _build_test_csv(tmp_dir / "1.csv", n_rows=4, prefix="b")
        total_expected = expected_kept_0 + expected_kept_1

        test_cfg = make_config(
            input_path=tmp_dir, output_path=tmp_dir / "out",
            csv_sep=",", sample_size=100, chunksize=3,
        )

        # ---- Part 1: run just far enough to process file 0, then simulate
        # a hard crash partway through file 1, and confirm partial progress
        # survived to disk. ----
        original_convert = convert_to_schema
        crash_marker = {"triggered": False}

        def crashy_convert(raw_row, process_id, cfg):
            if raw_row["name"].startswith("Test Process b2") and not crash_marker["triggered"]:
                crash_marker["triggered"] = True
                raise KeyboardInterrupt("Simulated crash for self-test")
            return original_convert(raw_row, process_id, cfg)

        globals()["convert_to_schema"] = crashy_convert
        crashed = False
        try:
            process_dataset(test_cfg, resume=True)
        except KeyboardInterrupt:
            crashed = True
        finally:
            globals()["convert_to_schema"] = original_convert

        checks.append(("simulated crash actually triggered", crashed))
        checkpoint_path = test_cfg.output_path / "pipeline_checkpoint.pkl"
        checks.append(("checkpoint file exists after crash", checkpoint_path.exists()))

        slots_dir = test_cfg.output_path / "_reservoir_slots"
        slots_before_resume = list(slots_dir.glob("slot_*.json")) if slots_dir.exists() else []
        checks.append(("some records were saved to disk BEFORE the crash",
                        len(slots_before_resume) > 0))

        # ---- Part 2: resume and confirm it completes correctly ----
        report = run_pipeline(test_cfg, resume=True)

        checks.append(("pipeline returned a non-empty report after resuming", bool(report)))
        stats = report.get("scan_and_conversion_stats", {})
        checks.append(("eligible_total matches expected kept rows across both files",
                        stats.get("eligible_total") == total_expected))
        checks.append(("wrong_language filter caught the mislabeled Spanish row",
                        stats.get("wrong_language", 0) >= 1))
        checks.append(("excluded_variant filter caught the choreography row",
                        stats.get("excluded_variant", 0) == 1))
        checks.append(("bad_json filter caught the malformed + non-BPMN rows",
                        stats.get("bad_json", 0) == 2))
        checks.append(("zero conversion/validation failures on valid rows",
                        stats.get("conversion_failures", 0) == 0
                        and stats.get("validation_failures", 0) == 0))
        checks.append(("train + eval counts match eligible total (sample_size > eligible)",
                        report["split_counts"]["total"] == total_expected))

        checks.append(("checkpoint cleaned up after successful completion",
                        not checkpoint_path.exists()))
        checks.append(("temp slots dir cleaned up after successful completion",
                        not slots_dir.exists()))

        train_dir = test_cfg.output_path / test_cfg.train_dirname
        eval_dir = test_cfg.output_path / test_cfg.eval_dirname
        saved_files = list(train_dir.glob("*.json")) + list(eval_dir.glob("*.json"))
        checks.append(("output JSON files exist on disk", len(saved_files) == total_expected))

        if saved_files:
            with open(saved_files[0], encoding="utf-8") as f:
                sample_record = json.load(f)
            checks.append(("saved record parses back as valid JSON with required fields",
                            all(k in sample_record for k in REQUIRED_TOP_LEVEL)))
            checks.append(("saved record's bpmn_xml is well-formed XML",
                            _xml_is_valid(sample_record["bpmn_xml"])))

        metadata_path = test_cfg.output_path / test_cfg.metadata_filename
        stats_path = test_cfg.output_path / test_cfg.stats_filename
        checks.append(("metadata.json was written", metadata_path.exists()))
        checks.append(("stats_report.json was written", stats_path.exists()))

        # ---- Part 3: memory footprint sanity check on a fresh, larger run ----
        import tracemalloc
        mem_cfg = make_config(
            input_path=tmp_dir / "0.csv", output_path=tmp_dir / "out_mem",
            csv_sep=",", sample_size=1000, chunksize=3,
        )
        tracemalloc.start()
        run_pipeline(mem_cfg, resume=False)
        _, peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        checks.append((f"peak memory stayed low ({peak/1024:.1f} KB) — no unbounded record accumulation",
                        peak < 20 * 1024 * 1024))

    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)

    print("=" * 60)
    print("SELF-TEST RESULTS")
    print("=" * 60)
    all_passed = True
    for description, passed in checks:
        status = "PASS" if passed else "FAIL"
        if not passed:
            all_passed = False
        print(f"  [{status}] {description}")
    print("=" * 60)
    print("ALL CHECKS PASSED" if all_passed else "SOME CHECKS FAILED — see above")
    print("=" * 60)
    return all_passed


self_test_passed = run_self_test()


06:17:17 | INFO     | Found 2 CSV file(s) to process.


Processing 0.csv: 0chunk [00:00, ?chunk/s]

06:17:18 | INFO     | Finished 0.csv            | scanned=9 passed_filter=5 eligible=5 reservoir=5/100


Processing 1.csv: 0chunk [00:00, ?chunk/s]

06:17:18 | INFO     | Resuming: 1 file(s) already done, 5 eligible records seen so far, reservoir filled 5/100.
06:17:18 | INFO     | Skipping 1 already-processed file(s).


...

Processing 0.csv: 0chunk [00:00, ?chunk/s]

06:17:18 | INFO     | Finished 0.csv            | scanned=9 passed_filter=5 eligible=5 reservoir=5/1000
06:17:18 | WARNING  | Only 5 eligible records found across the whole dataset (< requested sample_size=1000).
06:17:18 | INFO     | Processing complete: {'scanned': 9, 'bad_json': 2, 'wrong_notation': 2, 'wrong_language': 1, 'excluded_variant': 1, 'size_out_of_range': 0, 'passed_filter': 5, 'conversion_failures': 0, 'validation_failures': 0, 'eligible_total': 5, 'reservoir_size': 5}


Saving train:   0%|          | 0/4 [00:00<?, ?file/s]

Saving eval:   0%|          | 0/1 [00:00<?, ?file/s]

PIPELINE SUMMARY
Rows scanned:              9
Passed BPMN/language/size filters: 5
  Conversion failures:     0
  Validation failures:     0
Eligible (converted+valid): 5
Reservoir sample size:     5
Final train / eval split:  4 / 1
Avg tasks per process:     3.4
Avg gateways per process:  1.0
Avg task time (min):       129.47

Full report written to: C:\Users\yousu\AppData\Local\Temp\bpmn_pipeline_selftest_3t4w6day\out_mem\stats_report.json
SELF-TEST RESULTS
  [PASS] simulated crash actually triggered
  [PASS] checkpoint file exists after crash
  [PASS] some records were saved to disk BEFORE the crash
  [PASS] pipeline returned a non-empty report after resuming
  [PASS] eligible_total matches expected kept rows across both files
  [PASS] wrong_language filter caught the mislabeled Spanish row
  [PASS] excluded_variant filter caught the choreography row
  [PASS] bad_json filter caught the malformed + non-BPMN rows
  [PASS] zero conversion/validation failures on valid rows
  [PASS] trai


## 11. Run on Real Data

Once the self-test above passes, run the real pipeline against your actual
dataset. This scans every CSV under `CONFIG.input_path` — for the full
SAP-SAM export (~103 files, ~40GB), expect this to take a while (order of
tens of minutes to an hour depending on disk speed), since every row's JSON
is parsed and every diagram is walked to count tasks.


In [24]:

assert self_test_passed, "Self-test failed — fix the pipeline before running on real data."

report = run_pipeline(CONFIG)


06:17:20 | INFO     | Found 103 CSV file(s) to process.


Processing 0.csv: 0chunk [00:00, ?chunk/s]

06:18:12 | INFO     | Finished 0.csv            | scanned=10000 passed_filter=2401 eligible=2401 reservoir=2401/3000


Processing 10000.csv: 0chunk [00:00, ?chunk/s]

06:19:04 | INFO     | Finished 10000.csv        | scanned=20000 passed_filter=4861 eligible=4861 reservoir=3000/3000


...

Processing 990000.csv: 0chunk [00:00, ?chunk/s]

07:41:23 | INFO     | Finished 990000.csv       | scanned=1021471 passed_filter=250218 eligible=250218 reservoir=3000/3000
07:41:23 | INFO     | Processing complete: {'scanned': 1021471, 'bad_json': 405180, 'wrong_notation': 405180, 'wrong_language': 351315, 'excluded_variant': 1771, 'size_out_of_range': 12987, 'passed_filter': 250218, 'conversion_failures': 0, 'validation_failures': 0, 'eligible_total': 250218, 'reservoir_size': 3000}


Saving train:   0%|          | 0/2400 [00:00<?, ?file/s]

Saving eval:   0%|          | 0/600 [00:00<?, ?file/s]

PIPELINE SUMMARY
Rows scanned:              1021471
Passed BPMN/language/size filters: 250218
  Conversion failures:     0
  Validation failures:     0
Eligible (converted+valid): 250218
Reservoir sample size:     3000
Final train / eval split:  2400 / 600
Avg tasks per process:     7.29
Avg gateways per process:  3.71
Avg task time (min):       122.44

Full report written to: C:\Users\yousu\Downloads\SAP\sap_sam_2022\processed\stats_report.json



## 12. Notes

- **Point `CONFIG.input_path`** at your SAP-SAM CSV directory (the folder
  containing `0.csv`, `10000.csv`, ...) before running Section 11.
- **Language filtering** cross-checks the diagram's own `properties.language`
  field against `langdetect` on the name/description text — trusting the
  declared field alone lets mislabeled rows through.
- **Synthetic fields** (company, job, cost, timing, RACI, gateway
  probabilities) are generated with a per-record seed
  (`cfg.synth_seed XOR process_id`), so re-running reproduces byte-identical
  synthetic data.
- **`bpmn_xml`** is a simplified structural re-export — valid, parseable
  BPMN 2.0 XML sufficient for flow analysis, not a pixel-perfect
  reconstruction of the original Signavio diagram.
- All error handling favors **skip-and-log** over **crash-the-run**.
- The **self-test in Section 10** is the fast way to verify any future code
  changes didn't break the pipeline — it runs in well under a second and
  needs no real data.
- To adjust the split ratio, sample size, or filters, edit the `Config`
  dataclass in Section 1 only.
